# C11-neural-training — Practice p14 — Solution


**Type:** integrative (parts consume earlier results) · **Difficulty:** core · **Concepts:** softmax, cross-entropy-loss


The same shifted logits drive probabilities, stable log-sum-exp loss, and the
fused batch-mean gradient.


In [ ]:
import numpy as np

def softmax_ce_with_gradient(logits, labels):
    z=np.asarray(logits,dtype=np.float64); y=np.asarray(labels)
    if z.ndim!=2 or y.ndim!=1 or z.shape[0]!=y.shape[0] or z.shape[0]==0 or z.shape[1]==0:
        raise ValueError("invalid dimensions")
    if not np.issubdtype(y.dtype,np.integer) or np.any(y<0) or np.any(y>=z.shape[1]):
        raise ValueError("invalid labels")
    shifted=z-z.max(axis=1,keepdims=True); exp_shifted=np.exp(shifted); sums=exp_shifted.sum(axis=1,keepdims=True)
    probabilities=exp_shifted/sums
    loss=float(np.mean(np.log(sums[:,0])-shifted[np.arange(z.shape[0]),y]))
    gradient=probabilities.copy(); gradient[np.arange(z.shape[0]),y]-=1.0; gradient/=z.shape[0]
    return {"probabilities":probabilities,"loss":loss,"gradient":gradient}

logits_p14 = np.array([[10000.0, 9998.0, 9999.0], [-9000.0, -8997.0, -9001.0], [0.5, -0.25, 1.5]])
labels_p14 = np.array([0, 1, 2], dtype=int)
result_p14 = softmax_ce_with_gradient(logits_p14, labels_p14)


### Answer check


In [ ]:
expected_loss_p14 = 0.3021151116935261
expected_probabilities_p14 = np.array([
    [0.6652409557748218, 0.09003057317038046, 0.24472847105479764],
    [0.0466126225779739, 0.9362395518765058, 0.01714782554552039],
    [0.23862655824004825, 0.11271920470830459, 0.6486542370516473],
])
expected_gradient_p14 = np.array([
    [-0.1115863480750594, 0.03001019105679349, 0.08157615701826589],
    [0.01553754085932463, -0.02125348270783141, 0.0057159418485068],
    [0.07954218608001608, 0.03757306823610153, -0.11711525431611758],
])
assert np.isclose(result_p14["loss"], expected_loss_p14, atol=1e-11, rtol=1e-9)
assert np.allclose(result_p14["probabilities"], expected_probabilities_p14, atol=1e-11, rtol=1e-9)
assert np.allclose(result_p14["gradient"], expected_gradient_p14, atol=1e-11, rtol=1e-9)
assert np.all(np.isfinite(result_p14["probabilities"])) and np.isfinite(result_p14["loss"])
assert np.allclose(result_p14["probabilities"].sum(1),1,atol=1e-11,rtol=1e-9)
assert np.allclose(result_p14["gradient"].sum(1),0,atol=1e-11,rtol=1e-9)
for i_p14,j_p14 in [(0,0),(1,2),(2,1)]:
    delta_p14=np.zeros_like(logits_p14); delta_p14[i_p14,j_p14]=1e-6
    numeric_p14=(softmax_ce_with_gradient(logits_p14+delta_p14,labels_p14)["loss"]-softmax_ce_with_gradient(logits_p14-delta_p14,labels_p14)["loss"])/(2e-6)
    assert np.isclose(numeric_p14,result_p14["gradient"][i_p14,j_p14],atol=2e-6,rtol=2e-5)
